In [ ]:
'''import random
import numpy as np
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from agent import FloodWarningEnv

RUN_ID = "vecnorm"

# Seeds
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Training environment
env = make_vec_env(FloodWarningEnv, n_envs=8, seed=SEED)
env = VecNormalize(env, norm_obs=True, norm_reward=False)

# Evaluation environment
eval_env = make_vec_env(FloodWarningEnv, n_envs=1, seed=SEED)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

# Agent setup
model = PPO(
    "MlpPolicy",# MLP as the policy, takes observation vector as input and outputs probability distribution over the 4 warning levels
    env,
    learning_rate=3e-4, # how fast network weights update
    n_steps=2048, # how many steps collected across all environments before a policy update (total experience per update = 2048 * 8 = 16384 samples)
    batch_size=64, # how many samples used in each gradient update within a single policy update cycle
    n_epochs=10, # how many times collected experience is reused for gradient updates before being discarded
    gamma=0.99, # discount factor for future rewards
    verbose=1, # prints training progress
    seed=SEED, # seed for reproducibility
    tensorboard_log=f"./logs/{RUN_ID}/"
)

# Evaluation and model saving during training
eval_callback = EvalCallback( # periodically pauses training, runs agent deterministically on separate evaluation environment, saves best performing model
    eval_env,
    best_model_save_path=f"./models/{RUN_ID}/best_model",
    log_path=f"./logs/{RUN_ID}/",
    eval_freq=10_000, # evaluation runs every 10000 steps
    n_eval_episodes=100, # number of episodes to evaluate over for stable performance estimate
    deterministic=True,
)

# Executes training, with evaluation running at specified frequency
model.learn(
    total_timesteps=10_000_000, # runs training loop for n environment steps
    callback=eval_callback
)

# Save final model weights at end of training for resuming training if needed
model.save(f"./models/{RUN_ID}/final_model")'''

ValueError: Policy MlpLstmPolicy unknown

In [ ]:
'''import os
import random
import numpy as np
import torch
import optuna
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from agent import FloodWarningEnv

SEED = 1
N_TRIALS_PER_RUN = 25  # trials to add each time the script is run
TRAIN_STEPS = 500_000
N_EVAL_EPISODES = 100

STUDY_NAME = "flood_warning_tuning_vecnorm"
OUTPUT_DIR = "./tuning/tuning_vecnorm/"
DB_PATH = "sqlite:///./tuning/tuning_vecnorm/optuna_study_vecnorm.db"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def make_env(false_weight=0.3, missed_weight=2.0, jump_weight=0.5):
    return lambda: FloodWarningEnv(false_weight=false_weight, missed_weight=missed_weight, jump_weight=jump_weight)

def objective(trial):
    # PPO hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_epochs = trial.suggest_int("n_epochs", 5, 20)
    ent_coef = trial.suggest_float("ent_coef", 1e-4, 0.1, log=True)

    # Reward weights
    false_weight = trial.suggest_float("false_weight", 0.1, 1.0)
    missed_weight = trial.suggest_float("missed_weight", 1.0, 5.0)
    jump_weight = trial.suggest_float("jump_weight", 0.0, 1.0)

    # Set seeds
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # Training environment with VecNormalize
    env = make_vec_env(make_env(false_weight, missed_weight, jump_weight), n_envs=8, seed=SEED)
    env = VecNormalize(env, norm_obs=True, norm_reward=False)

    # Evaluation environment with VecNormalize, training=False to not update stats
    eval_env = make_vec_env(make_env(false_weight, missed_weight, jump_weight), n_envs=1, seed=SEED)
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

    # Agent
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        ent_coef=ent_coef,
        gamma=0.99,
        verbose=0,
        seed=SEED,
    )

    # Evaluation callback
    eval_callback = EvalCallback(
        eval_env,
        eval_freq=10_000,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
        verbose=0,
    )

    model.learn(total_timesteps=TRAIN_STEPS, callback=eval_callback)

    mean_reward = eval_callback.last_mean_reward
    return mean_reward

if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        storage=DB_PATH,        # saves/loads study to .db
        study_name=STUDY_NAME,
        load_if_exists=True     # resume if already exists
    )
    study.optimize(objective, n_trials=N_TRIALS_PER_RUN)

    print("Best trial:")
    print(f"  Mean reward: {study.best_trial.value:.4f}")
    print(f"  Params: {study.best_trial.params}")

    # Saves study results to .csv
    study.trials_dataframe().to_csv(os.path.join(OUTPUT_DIR, "study_results.csv"), index=False)
    print("Study results saved to " + os.path.join(OUTPUT_DIR, "study_results.csv"))'''

c:\Users\W L\Documents\Flood-RL-agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-28 08:33:30,234] Using an existing study with name 'flood_warning_tuning_vecnorm' instead of creating a new one.
[I 2026-03-28 09:03:06,995] Trial 70 finished with value: -1.6653938599999998 and parameters: {'learning_rate': 0.0004678298607999624, 'n_steps': 4096, 'batch_size': 64, 'n_epochs': 14, 'ent_coef': 0.0003589830004155226, 'false_weight': 0.1312872876614476, 'missed_weight': 1.9788478675270034, 'jump_weight': 0.8347022371664489}. Best is trial 45 with value: 3.4138379100000003.
[I 2026-03-28 09:32:22,113] Trial 71 finished with value: 2.21731131 and parameters: {'learning_rate': 0.00036260446092669784, 'n_steps': 4096, 'batch_size': 64, 'n_epochs': 12, 'ent_coef': 0.0018377034574190217, 'false_weight': 0.

Best trial:
  Mean reward: 3.4138
  Params: {'learning_rate': 0.0003726711992552962, 'n_steps': 1024, 'batch_size': 64, 'n_epochs': 10, 'ent_coef': 0.011768042239313008, 'false_weight': 0.26890112994618115, 'missed_weight': 1.0102250722070014, 'jump_weight': 0.848168848865283}
Study results saved to ./tuning/tuning_vecnorm/study_results.csv


In [ ]:
import os
import random
import numpy as np
import torch
import optuna
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from agent import FloodWarningEnv

SEED = 1
N_TRIALS_PER_RUN = 25
TRAIN_STEPS = 500_000
N_EVAL_EPISODES = 100

STUDY_NAME = "flood_warning_tuning_lstm_vecnorm"
OUTPUT_DIR = "./tuning/tuning_lstm_vecnorm/"
DB_PATH = "sqlite:///./tuning/tuning_lstm_vecnorm/optuna_study_lstm_vecnorm.db"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def make_env(false_weight=0.3, missed_weight=2.0, jump_weight=0.5):
    return lambda: FloodWarningEnv(false_weight=false_weight, missed_weight=missed_weight, jump_weight=jump_weight)

def objective(trial):
    # PPO hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [1024,2048, 4096])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_epochs = trial.suggest_int("n_epochs", 5, 20)
    ent_coef = trial.suggest_float("ent_coef", 1e-4, 0.1, log=True)

    # Reward weights
    false_weight = trial.suggest_float("false_weight", 0.1, 1.0)
    missed_weight = trial.suggest_float("missed_weight", 1.0, 5.0)
    jump_weight = trial.suggest_float("jump_weight", 0.0, 1.0)

    # Set seeds
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # Training environment with VecNormalize
    env = make_vec_env(make_env(false_weight, missed_weight, jump_weight), n_envs=8, seed=SEED)
    env = VecNormalize(env, norm_obs=True, norm_reward=False)

    # Evaluation environment with VecNormalize, training=False to not update stats
    eval_env = make_vec_env(make_env(false_weight, missed_weight, jump_weight), n_envs=1, seed=SEED)
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

    # Agent
    model = RecurrentPPO(
        "MlpLstmPolicy",   # LSTM policy, maintains hidden state across timesteps within an episode
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        ent_coef=ent_coef,
        gamma=0.99,
        verbose=0,
        seed=SEED,
    )

    # Evaluation callback
    eval_callback = EvalCallback(
        eval_env,
        eval_freq=10_000,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
        verbose=0,
    )

    model.learn(total_timesteps=TRAIN_STEPS, callback=eval_callback)

    mean_reward = eval_callback.last_mean_reward
    return mean_reward

if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        storage=DB_PATH,
        study_name=STUDY_NAME,
        load_if_exists=True
    )
    study.optimize(objective, n_trials=N_TRIALS_PER_RUN)

    print("Best trial:")
    print(f"  Mean reward: {study.best_trial.value:.4f}")
    print(f"  Params: {study.best_trial.params}")

    study.trials_dataframe().to_csv(os.path.recovery(OUTPUT_DIR, "study_results.csv"), index=False)
    print("Study results saved to " + os.path.join(OUTPUT_DIR, "study_results.csv"))

c:\Users\W L\Documents\Flood-RL-agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-31 10:53:29,057] Using an existing study with name 'flood_warning_tuning_lstm_vecnorm' instead of creating a new one.
[I 2026-03-31 11:53:51,873] Trial 25 finished with value: -2.37013068 and parameters: {'learning_rate': 7.253797546729923e-05, 'n_steps': 4096, 'batch_size': 32, 'n_epochs': 5, 'ent_coef': 0.0001080081052052096, 'false_weight': 0.21758692091448084, 'missed_weight': 1.960926845946016, 'jump_weight': 0.8451339288575543}. Best is trial 11 with value: 2.1031934100000003.
[I 2026-03-31 12:44:14,511] Trial 26 finished with value: -0.86268844 and parameters: {'learning_rate': 2.6985404857766002e-05, 'n_steps': 2048, 'batch_size': 64, 'n_epochs': 7, 'ent_coef': 0.00048777315327513786, 'false_weight': 0.108

In [2]:
import random
import numpy as np
import torch
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

from agent import FloodWarningEnv

SEED = 1
TOP_N = 5
TRAIN_STEPS = 1_000_000  # full training run
N_EVAL_EPISODES = 100
OUTPUT_DIR = "./models/top_runs/"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def make_env(false_weight, missed_weight, jump_weight):
    return lambda: FloodWarningEnv(false_weight=false_weight, missed_weight=missed_weight, jump_weight=jump_weight)

def train_top_configs(study_csv, run_id):
    # Load and sort by mean reward
    df = pd.read_csv(study_csv)
    df = df.dropna(subset=["value"])
    top_configs = df.nlargest(TOP_N, "value")

    for rank, (_, row) in enumerate(top_configs.iterrows(), start=1):
        print(f"\n=== Run {run_id} | Top {rank} config | Mean reward: {row['value']:.4f} ===")

        # Extract hyperparameters
        learning_rate = row["params_learning_rate"]
        n_steps = int(row["params_n_steps"])
        batch_size   = int(row["params_batch_size"])
        n_epochs = int(row["params_n_epochs"])
        ent_coef = row["params_ent_coef"]
        false_weight = row["params_false_weight"]
        missed_weight = row["params_missed_weight"]
        jump_weight = row["params_jump_weight"]

        # Set seeds
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)

        # Environments
        env = make_vec_env(make_env(false_weight, missed_weight, jump_weight), n_envs=8, seed=SEED)
        eval_env = FloodWarningEnv(false_weight=false_weight, missed_weight=missed_weight, jump_weight=jump_weight)
        eval_env.reset(seed=SEED)

        # Model save path
        model_dir = os.path.join(OUTPUT_DIR, f"{run_id}_rank{rank}/")
        os.makedirs(model_dir, exist_ok=True)

        # Agent
        model = PPO(
            "MlpPolicy",
            env,
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            n_epochs=n_epochs,
            ent_coef=ent_coef,
            gamma=0.99,
            verbose=1,
            seed=SEED,
        )

        # Evaluation callback
        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path=os.path.join(model_dir, "best_model/"),
            log_path=os.path.join(model_dir, "logs/"),
            eval_freq=10_000,
            n_eval_episodes=N_EVAL_EPISODES,
            deterministic=True,
            verbose=1,
        )

        model.learn(total_timesteps=TRAIN_STEPS, callback=eval_callback)

        # Save final model
        model.save(os.path.join(model_dir, "final_model"))

        # Save config used
        import json
        with open(os.path.join(model_dir, "config.json"), "w") as f:
            json.dump({
                "rank": rank,
                "tuning_mean_reward": row["value"],
                "learning_rate": learning_rate,
                "n_steps": n_steps,
                "batch_size": batch_size,
                "n_epochs": n_epochs,
                "ent_coef": ent_coef,
                "false_weight": false_weight,
                "missed_weight": missed_weight,
                "jump_weight": jump_weight,
            }, f, indent=4)

        print(f"Saved to {model_dir}")

if __name__ == "__main__":
    #train_top_configs("./tuning/tuning/study_results.csv", run_id="nonorm")
    train_top_configs("./tuning/tuning_vecnorm/study_results.csv", run_id="vecnorm")


=== Run vecnorm | Top 1 config | Mean reward: 4.8972 ===
Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean     | -17.1    |
| time/              |          |
|    fps             | 366      |
|    iterations      | 1        |
|    time_elapsed    | 22       |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 10          |
|    ep_rew_mean          | -13.8       |
| time/                   |             |
|    fps                  | 341         |
|    iterations           | 2           |
|    time_elapsed         | 47          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.020273194 |
|    clip_fraction        | 0.46        |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.37      

c:\Users\W L\Documents\Flood-RL-agent\venv\Lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=80000, episode_reward=0.95 +/- 17.44
Episode length: 10.00 +/- 0.00
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 10          |
|    mean_reward          | 0.951       |
| time/                   |             |
|    total_timesteps      | 80000       |
| train/                  |             |
|    approx_kl            | 0.005258832 |
|    clip_fraction        | 0.0455      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.517      |
|    explained_variance   | 0.078       |
|    learning_rate        | 0.000159    |
|    loss                 | 23.9        |
|    n_updates            | 45          |
|    policy_gradient_loss | -0.00256    |
|    value_loss           | 54.3        |
-----------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean     | -2.1     |

c:\Users\W L\Documents\Flood-RL-agent\venv\Lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=160000, episode_reward=4.12 +/- 11.22
Episode length: 10.00 +/- 0.00
-------------------------------------------
| eval/                   |               |
|    mean_ep_length       | 10            |
|    mean_reward          | 4.12          |
| time/                   |               |
|    total_timesteps      | 160000        |
| train/                  |               |
|    approx_kl            | 0.00031211073 |
|    clip_fraction        | 0.00242       |
|    clip_range           | 0.2           |
|    entropy_loss         | -0.161        |
|    explained_variance   | 0.301         |
|    learning_rate        | 0.000159      |
|    loss                 | 33.1          |
|    n_updates            | 95            |
|    policy_gradient_loss | -0.00011      |
|    value_loss           | 72.2          |
-------------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10     

...

c:\Users\W L\Documents\Flood-RL-agent\venv\Lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=880000, episode_reward=1.98 +/- 15.98
Episode length: 10.00 +/- 0.00
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 10          |
|    mean_reward          | 1.98        |
| time/                   |             |
|    total_timesteps      | 880000      |
| train/                  |             |
|    approx_kl            | 4.34494e-06 |
|    clip_fraction        | 0           |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.0343     |
|    explained_variance   | 0.85        |
|    learning_rate        | 0.000126    |
|    loss                 | 22.2        |
|    n_updates            | 535         |
|    policy_gradient_loss | -1.48e-05   |
|    value_loss           | 68          |
-----------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean     | 3.48     |
| time/             

c:\Users\W L\Documents\Flood-RL-agent\venv\Lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=960000, episode_reward=2.79 +/- 12.82
Episode length: 10.00 +/- 0.00
-------------------------------------------
| eval/                   |               |
|    mean_ep_length       | 10            |
|    mean_reward          | 2.79          |
| time/                   |               |
|    total_timesteps      | 960000        |
| train/                  |               |
|    approx_kl            | 4.5654204e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -0.0341       |
|    explained_variance   | 0.868         |
|    learning_rate        | 0.000126      |
|    loss                 | 62            |
|    n_updates            | 585           |
|    policy_gradient_loss | -0.000116     |
|    value_loss           | 60.7          |
-------------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean  

KeyboardInterrupt: 